In [7]:
# ============================================================
# CELDA 1 — IMPORTS Y CONFIGURACIÓN DEL PROYECTO
# ============================================================

from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)


# ============================================================
# LOCALIZAR AUTOMÁTICAMENTE LA RAÍZ DEL PROYECTO
# ============================================================

CURRENT = Path.cwd().resolve()

candidatos = [
    CURRENT,
    CURRENT.parent,
    CURRENT.parent.parent,
]

ROOT = None

for candidato in candidatos:
    if (
        (candidato / "techmind").exists()
        and (candidato / "pyproject.toml").exists()
    ):
        ROOT = candidato
        break


if ROOT is None:
    raise FileNotFoundError(
        "No se pudo localizar la raíz de TechMind. "
        "Debe existir una carpeta 'techmind/' y un archivo "
        "'pyproject.toml'."
    )


# ============================================================
# AGREGAR LA RAÍZ AL PYTHONPATH
# ============================================================

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


# ============================================================
# IMPORTAR TECHMIND
# ============================================================

from techmind import TechMindPredictor


# ============================================================
# INFORMACIÓN DE CONTROL
# ============================================================

print("=" * 70)
print("TECHMIND — EVALUACIÓN MULTILINGÜE")
print("=" * 70)

print("Directorio actual:")
print(CURRENT)

print("\nRaíz detectada:")
print(ROOT)

print("\nPaquete TechMind:")
print(ROOT / "techmind")

print("\nImport TechMindPredictor: OK")

TECHMIND — EVALUACIÓN MULTILINGÜE
Directorio actual:
C:\Users\MAMÁ\Downloads\techmind-v2\notebooks

Raíz detectada:
C:\Users\MAMÁ\Downloads\techmind-v2

Paquete TechMind:
C:\Users\MAMÁ\Downloads\techmind-v2\techmind

Import TechMindPredictor: OK


In [8]:
# ============================================================
# CELDA 2 — RUTAS DEL BENCHMARK MULTILINGÜE
# ============================================================

from pathlib import Path

# ROOT ya fue detectado en la Celda 1

DATA_PATH = (
    ROOT
    / "data"
    / "evaluation"
    / "multilingual_benchmark.csv"
)

REPORT_DIR = (
    ROOT
    / "reports"
    / "multilingual"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("RUTAS — EVALUACIÓN MULTILINGÜE")
print("=" * 70)

print("\nDataset:")
print(DATA_PATH)

print("\nExiste dataset:")
print(DATA_PATH.exists())

print("\nDirectorio de reportes:")
print(REPORT_DIR)

print("\nExiste directorio de reportes:")
print(REPORT_DIR.exists())

RUTAS — EVALUACIÓN MULTILINGÜE

Dataset:
C:\Users\MAMÁ\Downloads\techmind-v2\data\evaluation\multilingual_benchmark.csv

Existe dataset:
True

Directorio de reportes:
C:\Users\MAMÁ\Downloads\techmind-v2\reports\multilingual

Existe directorio de reportes:
True


In [9]:
# ============================================================
# CELDA 3 — CARGA DEL BENCHMARK
# ============================================================

df_multi = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig"
)

print("=" * 70)
print("BENCHMARK MULTILINGÜE")
print("=" * 70)

print(f"\nRegistros: {len(df_multi)}")

print("\nColumnas:")
print(df_multi.columns.tolist())

print("\nIdiomas:")
print(df_multi["idioma"].value_counts().sort_index())

print("\nCategorías:")
print(df_multi["categoria_real"].value_counts().sort_index())

print("\nDificultad:")
print(df_multi["dificultad"].value_counts())

print("\nOrigen:")
print(df_multi["origen"].value_counts())

print("\nPrimeros registros:")
display(df_multi.head(8))

BENCHMARK MULTILINGÜE

Registros: 80

Columnas:
['case_id', 'idioma', 'categoria_real', 'texto', 'dificultad', 'origen']

Idiomas:
idioma
en       20
es       20
es_en    20
ru       20
Name: count, dtype: int64

Categorías:
categoria_real
backend        20
cloud          20
datascience    20
frontend       20
Name: count, dtype: int64

Dificultad:
dificultad
facil      32
media      32
dificil    16
Name: count, dtype: int64

Origen:
origen
synthetic_controlled    80
Name: count, dtype: int64

Primeros registros:


,case_id,idioma,categoria_real,texto,dificultad,origen
0,backend_001,es,backend,Crear una API REST con Spring Boot y Java usan...,facil,synthetic_controlled
1,backend_001,en,backend,Build a REST API with Spring Boot and Java usi...,facil,synthetic_controlled
2,backend_001,ru,backend,Создать REST API на Spring Boot и Java с контр...,facil,synthetic_controlled
3,backend_001,es_en,backend,Crear una REST API con Spring Boot y Java usin...,facil,synthetic_controlled
4,backend_002,es,backend,Implementar autenticación con JWT en Node.js y...,facil,synthetic_controlled
5,backend_002,en,backend,Implement JWT authentication in Node.js and Ex...,facil,synthetic_controlled
6,backend_002,ru,backend,Реализовать аутентификацию JWT в Node.js и Exp...,facil,synthetic_controlled
7,backend_002,es_en,backend,Implementar JWT authentication en Node.js y Ex...,facil,synthetic_controlled


In [10]:
# ============================================================
# CELDA 4 — BALANCE POR IDIOMA Y CATEGORÍA
# ============================================================

tabla_balance = pd.crosstab(
    df_multi["idioma"],
    df_multi["categoria_real"]
)

print("=" * 70)
print("BALANCE DEL BENCHMARK")
print("=" * 70)

display(tabla_balance)

balance_correcto = (
    tabla_balance
    .reindex(
        index=["es", "en", "ru", "es_en"],
        columns=[
            "backend",
            "cloud",
            "datascience",
            "frontend"
        ]
    )
    .fillna(0)
    .eq(5)
    .all()
    .all()
)

print("\nBalance esperado:")
print("5 casos por idioma × categoría")

print("\nBenchmark balanceado:")
print(balance_correcto)

BALANCE DEL BENCHMARK


categoria_real,backend,cloud,datascience,frontend
idioma,,,,
en,5,5,5,5
es,5,5,5,5
es_en,5,5,5,5
ru,5,5,5,5



Balance esperado:
5 casos por idioma × categoría

Benchmark balanceado:
True


In [11]:
# ============================================================
# CELDA 5 — CARGA Y VERIFICACIÓN DEL MODELO TECHMIND v1.1.0
# ============================================================

print("=" * 70)
print("CARGA DEL MODELO TECHMIND")
print("=" * 70)

predictor = TechMindPredictor(
    package_root=ROOT,
    verify_hash=True
)

health = predictor.health()

print("\n✅ Predictor cargado correctamente")

print("\nVersión del modelo:")
print(health.get("model_version"))

print("\nFeatures Word:")
print(health.get("word_features"))

print("\nFeatures Char:")
print(health.get("char_features"))

print("\nFeatures totales:")
print(health.get("total_features"))

print("\nSHA-256:")
print(health.get("model_sha256"))

CARGA DEL MODELO TECHMIND

✅ Predictor cargado correctamente

Versión del modelo:
1.1.0

Features Word:
30000

Features Char:
30000

Features totales:
60000

SHA-256:
756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6


In [12]:
# ============================================================
# CELDA 6 — VALIDACIÓN DE INTEGRIDAD
# ============================================================

EXPECTED_MODEL_VERSION = "1.1.0"

EXPECTED_SHA256 = (
    "756b2577e731336ead95852ee1d8d752408762478a23d0bbf42cc9537e136ff6"
)

EXPECTED_WORD_FEATURES = 30000
EXPECTED_CHAR_FEATURES = 30000
EXPECTED_TOTAL_FEATURES = 60000


validacion_modelo = {
    "version_correcta":
        health.get("model_version") == EXPECTED_MODEL_VERSION,

    "sha256_correcto":
        health.get("model_sha256") == EXPECTED_SHA256,

    "word_features_correctas":
        health.get("word_features") == EXPECTED_WORD_FEATURES,

    "char_features_correctas":
        health.get("char_features") == EXPECTED_CHAR_FEATURES,

    "total_features_correctas":
        health.get("total_features") == EXPECTED_TOTAL_FEATURES,
}


print("=" * 70)
print("VALIDACIÓN DEL MODELO")
print("=" * 70)

for nombre, resultado in validacion_modelo.items():
    estado = "✅" if resultado else "❌"
    print(f"{estado} {nombre}: {resultado}")


MODELO_VALIDO = all(
    validacion_modelo.values()
)

print("\n" + "-" * 70)

print(
    "MODELO LISTO PARA BENCHMARK:",
    MODELO_VALIDO
)

if not MODELO_VALIDO:
    raise RuntimeError(
        "El modelo cargado no coincide con TechMind v1.1.0 aprobado."
    )

VALIDACIÓN DEL MODELO
✅ version_correcta: True
✅ sha256_correcto: True
✅ word_features_correctas: True
✅ char_features_correctas: True
✅ total_features_correctas: True

----------------------------------------------------------------------
MODELO LISTO PARA BENCHMARK: True


In [13]:
# ============================================================
# CELDA 7 — SMOKE TEST MULTILINGÜE
# ============================================================

texto_prueba = df_multi.iloc[0]["texto"]

print("=" * 70)
print("SMOKE TEST")
print("=" * 70)

print("\nTexto:")
print(texto_prueba)

resultado_prueba = predictor.predict(
    [texto_prueba],
    include_explanation=False,
    top_k=4
)

print("\nResultado completo:")
resultado_prueba

SMOKE TEST

Texto:
Crear una API REST con Spring Boot y Java usando controladores, servicios y repositorios para gestionar usuarios y pedidos.

Resultado completo:


{'resumen': {'request_id': '8417ef3b-2df4-44e7-80b4-d1fcabaad6d0',
  'timestamp_utc': '2026-08-13T07:19:29.996+00:00',
  'interface_version': '1.0.0',
  'model_version': '1.1.0',
  'model_name': 'TF-IDF Word + Char 3-6 + SGDClassifier optimizado',
  'documents_received': 1,
  'documents_accepted': 1,
  'documents_review': 0,
  'documents_rejected': 0,
  'duration_seconds': 0.23180059995502234,
  'milliseconds_per_document': 231.80059995502234,
  'explanations_included': False,
  'margin_is_probability': False},
 'resultados': [{'request_id': '8417ef3b-2df4-44e7-80b4-d1fcabaad6d0',
   'record_id': 0,
   'timestamp_utc': '2026-08-13T07:19:29.996+00:00',
   'input_type': 'str',
   'text': 'Crear una API REST con Spring Boot y Java usando controladores, servicios y repositorios para gestionar usuarios y pedidos.',
   'characters': 123,
   'words': 19,
   'valid_input': True,
   'validation_message': 'Inferencia completada.',
   'estado': 'aceptada',
   'categoria_predicha': 'backend',
   '

In [14]:
# ============================================================
# CELDA 8 — INSPECCIÓN DEL CONTRATO DE PREDICCIÓN
# ============================================================

print("=" * 70)
print("CONTRATO DE PREDICCIÓN")
print("=" * 70)

print("\nClaves principales:")
print(
    list(resultado_prueba.keys())
)

if "resultados" in resultado_prueba:
    print("\nClaves de una predicción:")

    primera_prediccion = (
        resultado_prueba["resultados"][0]
    )

    for clave in primera_prediccion.keys():
        print(f" - {clave}")

CONTRATO DE PREDICCIÓN

Claves principales:
['resumen', 'resultados']

Claves de una predicción:
 - request_id
 - record_id
 - timestamp_utc
 - input_type
 - text
 - characters
 - words
 - valid_input
 - validation_message
 - estado
 - categoria_predicha
 - segunda_categoria
 - puntuacion_ganadora
 - puntuacion_segunda
 - margen_decision
 - nivel_margen
 - terminos_activos
 - word_features_activas
 - char_features_activas
 - features_activas_total
 - accion_recomendada
 - requiere_revision
 - prediccion_utilizable
 - advertencias
 - ranking_categorias
 - explicacion


In [15]:
# ============================================================
# CELDA 9 — INFERENCIA MULTILINGÜE COMPLETA
# ============================================================

print("=" * 70)
print("EJECUTANDO BENCHMARK MULTILINGÜE")
print("=" * 70)

textos = (
    df_multi["texto"]
    .astype(str)
    .tolist()
)

print(f"\nDocumentos a evaluar: {len(textos)}")

resultado_multi = predictor.predict(
    textos,
    include_explanation=False,
    top_k=4
)

print("\n✅ Inferencia completada")

print("\nClaves del resultado:")
print(
    list(resultado_multi.keys())
)

if "resumen" in resultado_multi:
    print("\nResumen:")
    print(resultado_multi["resumen"])

EJECUTANDO BENCHMARK MULTILINGÜE

Documentos a evaluar: 80

✅ Inferencia completada

Claves del resultado:
['resumen', 'resultados']

Resumen:
{'request_id': '9d81e0ea-4a1e-4bba-bb78-a4bf97d0bcb6', 'timestamp_utc': '2026-08-13T07:20:24.386+00:00', 'interface_version': '1.0.0', 'model_version': '1.1.0', 'model_name': 'TF-IDF Word + Char 3-6 + SGDClassifier optimizado', 'documents_received': 80, 'documents_accepted': 40, 'documents_review': 40, 'documents_rejected': 0, 'duration_seconds': 0.20077860006131232, 'milliseconds_per_document': 2.509732500766404, 'explanations_included': False, 'margin_is_probability': False}


In [16]:
# ============================================================
# CELDA 10 — DATAFRAME DE PREDICCIONES
# ============================================================

predicciones = pd.DataFrame(
    resultado_multi["resultados"]
)

print("=" * 70)
print("PREDICCIONES DEL MODELO")
print("=" * 70)

print(f"\nFilas: {len(predicciones)}")

print("\nColumnas:")
for columna in predicciones.columns:
    print(f" - {columna}")

print("\nPrimeras predicciones:")
display(
    predicciones.head(10)
)

PREDICCIONES DEL MODELO

Filas: 80

Columnas:
 - request_id
 - record_id
 - timestamp_utc
 - input_type
 - text
 - characters
 - words
 - valid_input
 - validation_message
 - estado
 - categoria_predicha
 - segunda_categoria
 - puntuacion_ganadora
 - puntuacion_segunda
 - margen_decision
 - nivel_margen
 - terminos_activos
 - word_features_activas
 - char_features_activas
 - features_activas_total
 - accion_recomendada
 - requiere_revision
 - prediccion_utilizable
 - advertencias
 - ranking_categorias
 - explicacion

Primeras predicciones:


,request_id,record_id,timestamp_utc,input_type,text,characters,words,valid_input,validation_message,estado,...,terminos_activos,word_features_activas,char_features_activas,features_activas_total,accion_recomendada,requiere_revision,prediccion_utilizable,advertencias,ranking_categorias,explicacion
0,9d81e0ea-4a1e-4bba-bb78-a4bf97d0bcb6,0,2026-08-13T07:20:24.386+00:00,str,Crear una API REST con Spring Boot y Java usan...,123,19,True,Inferencia completada.,aceptada,...,213,2,211,213,Predicción utilizable automáticamente,False,True,[],"[{'position': 1, 'category': 'backend', 'score...",None
1,9d81e0ea-4a1e-4bba-bb78-a4bf97d0bcb6,1,2026-08-13T07:20:24.386+00:00,str,Build a REST API with Spring Boot and Java usi...,116,19,True,Inferencia completada.,aceptada,...,249,11,238,249,Predicción utilizable automáticamente,False,True,[],"[{'position': 1, 'category': 'backend', 'score...",None
2,9d81e0ea-4a1e-4bba-bb78-a4bf97d0bcb6,2,2026-08-13T07:20:24.386+00:00,str,Создать REST API на Spring Boot и Java с контр...,123,18,True,Inferencia completada.,revision,...,60,3,57,60,Revisión humana recomendada,True,True,[Cobertura reducida de características.],"[{'position': 1, 'category': 'backend', 'score...",None
3,9d81e0ea-4a1e-4bba-bb78-a4bf97d0bcb6,3,2026-08-13T07:20:24.386+00:00,str,Crear una REST API con Spring Boot y Java usin...,115,19,True,Inferencia completada.,aceptada,...,247,5,242,247,Predicción utilizable automáticamente,False,True,[],"[{'position': 1, 'category': 'backend', 'score...",None
4,9d81e0ea-4a1e-4bba-bb78-a4bf97d0bcb6,4,2026-08-13T07:20:24.386+00:00,str,Implementar autenticación con JWT en Node.js y...,121,17,True,Inferencia completada.,aceptada,...,248,1,247,248,Predicción utilizable automáticamente,False,True,[],"[{'position': 1, 'category': 'backend', 'score...",None
5,9d81e0ea-4a1e-4bba-bb78-a4bf97d0bcb6,5,2026-08-13T07:20:24.386+00:00,str,Implement JWT authentication in Node.js and Ex...,110,15,True,Inferencia completada.,aceptada,...,274,9,265,274,Predicción utilizable automáticamente,False,True,[],"[{'position': 1, 'category': 'backend', 'score...",None
6,9d81e0ea-4a1e-4bba-bb78-a4bf97d0bcb6,6,2026-08-13T07:20:24.386+00:00,str,Реализовать аутентификацию JWT в Node.js и Exp...,118,15,True,Inferencia completada.,revision,...,118,3,115,118,Revisión humana recomendada,True,True,[Cobertura reducida de características.],"[{'position': 1, 'category': 'backend', 'score...",None
7,9d81e0ea-4a1e-4bba-bb78-a4bf97d0bcb6,7,2026-08-13T07:20:24.386+00:00,str,Implementar JWT authentication en Node.js y Ex...,110,15,True,Inferencia completada.,aceptada,...,268,4,264,268,Predicción utilizable automáticamente,False,True,[],"[{'position': 1, 'category': 'backend', 'score...",None
8,9d81e0ea-4a1e-4bba-bb78-a4bf97d0bcb6,8,2026-08-13T07:20:24.386+00:00,str,Construir un servicio con Django REST Framewor...,130,18,True,Inferencia completada.,revision,...,259,2,257,259,Revisión humana recomendada,True,True,[El margen de decisión es reducido.],"[{'position': 1, 'category': 'backend', 'score...",None
9,9d81e0ea-4a1e-4bba-bb78-a4bf97d0bcb6,9,2026-08-13T07:20:24.386+00:00,str,"Build a service with Django REST Framework, se...",125,19,True,Inferencia completada.,revision,...,283,14,269,283,Revisión humana recomendada,True,True,[El margen de decisión es reducido.],"[{'position': 1, 'category': 'backend', 'score...",None


In [17]:
# ============================================================
# CELDA 11 — RESULTADOS CONSOLIDADOS
# ============================================================

df_resultados = pd.concat(
    [
        df_multi.reset_index(drop=True),
        predicciones.reset_index(drop=True)
    ],
    axis=1
)

# ------------------------------------------------------------
# Predicción correcta o incorrecta
# ------------------------------------------------------------

df_resultados["correcta"] = (
    df_resultados["categoria_real"]
    ==
    df_resultados["categoria_predicha"]
)

print("=" * 70)
print("RESULTADOS CONSOLIDADOS")
print("=" * 70)

print(f"\nDocumentos: {len(df_resultados)}")

print(
    "\nPredicciones correctas:",
    int(df_resultados["correcta"].sum())
)

print(
    "Predicciones incorrectas:",
    int((~df_resultados["correcta"]).sum())
)

print(
    "Accuracy inicial:",
    round(
        df_resultados["correcta"].mean(),
        4
    )
)

display(
    df_resultados[
        [
            "case_id",
            "idioma",
            "categoria_real",
            "categoria_predicha",
            "estado",
            "correcta"
        ]
    ].head(20)
)

RESULTADOS CONSOLIDADOS

Documentos: 80

Predicciones correctas: 69
Predicciones incorrectas: 11
Accuracy inicial: 0.8625


,case_id,idioma,categoria_real,categoria_predicha,estado,correcta
0,backend_001,es,backend,backend,aceptada,True
1,backend_001,en,backend,backend,aceptada,True
2,backend_001,ru,backend,backend,revision,True
3,backend_001,es_en,backend,backend,aceptada,True
4,backend_002,es,backend,backend,aceptada,True
5,backend_002,en,backend,backend,aceptada,True
6,backend_002,ru,backend,backend,revision,True
7,backend_002,es_en,backend,backend,aceptada,True
8,backend_003,es,backend,backend,revision,True
9,backend_003,en,backend,backend,revision,True


In [18]:
# ============================================================
# CELDA 12 — ACCURACY POR IDIOMA
# ============================================================

accuracy_idioma = (
    df_resultados
    .groupby("idioma")
    .agg(
        documentos=(
            "correcta",
            "size"
        ),
        correctas=(
            "correcta",
            "sum"
        ),
        accuracy=(
            "correcta",
            "mean"
        )
    )
    .reset_index()
)

accuracy_idioma["accuracy"] = (
    accuracy_idioma["accuracy"]
    .round(4)
)

print("=" * 70)
print("ACCURACY POR IDIOMA")
print("=" * 70)

display(
    accuracy_idioma
)

ACCURACY POR IDIOMA


,idioma,documentos,correctas,accuracy
0,en,20,19,0.95
1,es,20,15,0.75
2,es_en,20,19,0.95
3,ru,20,16,0.80


In [19]:
# ============================================================
# CELDA 13 — MÉTRICAS COMPLETAS POR IDIOMA
# ============================================================

metricas_idioma = []

for idioma, grupo in df_resultados.groupby("idioma"):

    y_true = grupo["categoria_real"]
    y_pred = grupo["categoria_predicha"]

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    metricas_idioma.append({
        "idioma": idioma,
        "documentos": len(grupo),
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    })


df_metricas_idioma = (
    pd.DataFrame(metricas_idioma)
    .sort_values(
        "f1_macro",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("MÉTRICAS MULTILINGÜES POR IDIOMA")
print("=" * 70)

display(
    df_metricas_idioma.round(4)
)

MÉTRICAS MULTILINGÜES POR IDIOMA


,idioma,documentos,accuracy,precision_macro,recall_macro,f1_macro
0,en,20,0.95,0.9583,0.95,0.9495
1,es_en,20,0.95,0.9583,0.95,0.9495
2,ru,20,0.80,0.8889,0.80,0.7619
3,es,20,0.75,0.7938,0.75,0.7465


In [20]:
# ============================================================
# CELDA 14 — RENDIMIENTO POR IDIOMA Y CATEGORÍA
# ============================================================

tabla_idioma_categoria = (
    df_resultados
    .groupby([
        "idioma",
        "categoria_real"
    ])
    ["correcta"]
    .mean()
    .unstack()
    .reindex(
        index=[
            "es",
            "en",
            "ru",
            "es_en"
        ],
        columns=[
            "backend",
            "cloud",
            "datascience",
            "frontend"
        ]
    )
)

print("=" * 70)
print("ACCURACY POR IDIOMA Y CATEGORÍA")
print("=" * 70)

display(
    tabla_idioma_categoria.round(2)
)

ACCURACY POR IDIOMA Y CATEGORÍA


categoria_real,backend,cloud,datascience,frontend
idioma,,,,
es,1.0,0.6,0.6,0.8
en,1.0,0.8,1.0,1.0
ru,1.0,1.0,0.2,1.0
es_en,1.0,0.8,1.0,1.0


In [21]:
# ============================================================
# CELDA 15 — ANÁLISIS DE ERRORES
# ============================================================

errores = (
    df_resultados[
        ~df_resultados["correcta"]
    ]
    .copy()
)

columnas_error = [
    "case_id",
    "idioma",
    "categoria_real",
    "categoria_predicha",
    "segunda_categoria",
    "estado",
    "margen_decision",
    "word_features_activas",
    "char_features_activas",
    "features_activas_total",
    "texto"
]

columnas_disponibles = [
    c
    for c in columnas_error
    if c in errores.columns
]

print("=" * 70)
print("PREDICCIONES INCORRECTAS")
print("=" * 70)

print(
    f"\nTotal errores: {len(errores)}"
)

display(
    errores[
        columnas_disponibles
    ].sort_values(
        [
            "idioma",
            "categoria_real"
        ]
    )
)

PREDICCIONES INCORRECTAS

Total errores: 11


,case_id,idioma,categoria_real,categoria_predicha,segunda_categoria,estado,margen_decision,word_features_activas,char_features_activas,features_activas_total,texto
37,cloud_005,en,cloud,frontend,cloud,revision,0.304943,19,321,340,Build a deployment pipeline that creates Docke...
20,cloud_001,es,cloud,backend,cloud,revision,0.315313,5,229,234,"Desplegar una aplicación en AWS usando ECR, EC..."
36,cloud_005,es,cloud,frontend,cloud,revision,0.156966,4,285,289,Construir un pipeline de despliegue que genere...
48,datascience_003,es,datascience,cloud,frontend,revision,0.084204,3,208,211,Entrenar una red neuronal convolucional con Te...
56,datascience_005,es,datascience,backend,cloud,revision,0.343937,5,258,263,"Reducir variables con PCA, aplicar K-Means par..."
76,frontend_005,es,frontend,backend,frontend,revision,0.111896,3,233,236,Crear una SPA en TypeScript que actualice el D...
39,cloud_005,es_en,cloud,frontend,cloud,revision,0.059941,9,284,293,Construir un deployment pipeline que build Doc...
46,datascience_002,ru,datascience,cloud,datascience,revision,0.216145,3,40,43,Предсказать отток клиентов с помощью логистиче...
50,datascience_003,ru,datascience,cloud,backend,revision,0.033715,6,81,87,Обучить сверточную нейронную сеть в TensorFlow...
54,datascience_004,ru,datascience,cloud,backend,revision,0.830423,2,11,13,"Проанализировать временной ряд продаж, изучить..."


In [22]:
# ============================================================
# CELDA 16 — ESTADOS OPERACIONALES POR IDIOMA
# ============================================================

tabla_estados = (
    pd.crosstab(
        df_resultados["idioma"],
        df_resultados["estado"]
    )
    .reindex(
        index=[
            "es",
            "en",
            "ru",
            "es_en"
        ],
        fill_value=0
    )
)

print("=" * 70)
print("ESTADOS OPERACIONALES POR IDIOMA")
print("=" * 70)

display(tabla_estados)

ESTADOS OPERACIONALES POR IDIOMA


estado,aceptada,revision
idioma,,
es,11,9
en,16,4
ru,0,20
es_en,13,7


In [23]:
tabla_estados_pct = (
    pd.crosstab(
        df_resultados["idioma"],
        df_resultados["estado"],
        normalize="index"
    )
    * 100
)

tabla_estados_pct = (
    tabla_estados_pct
    .reindex([
        "es",
        "en",
        "ru",
        "es_en"
    ])
)

print("\nPORCENTAJES:")

display(
    tabla_estados_pct.round(2)
)


PORCENTAJES:


estado,aceptada,revision
idioma,,
es,55.0,45.0
en,80.0,20.0
ru,0.0,100.0
es_en,65.0,35.0


In [24]:
# ============================================================
# CELDA 17 — ACCURACY DE PREDICCIONES ACEPTADAS
# ============================================================

resultados_aceptadas = []

for idioma, grupo in df_resultados.groupby("idioma"):

    aceptadas = grupo[
        grupo["estado"] == "aceptada"
    ]

    if len(aceptadas) > 0:
        accuracy_aceptadas = (
            aceptadas["correcta"]
            .mean()
        )
    else:
        accuracy_aceptadas = np.nan

    resultados_aceptadas.append({
        "idioma": idioma,
        "documentos": len(grupo),
        "aceptadas": len(aceptadas),
        "correctas_aceptadas": int(
            aceptadas["correcta"].sum()
        ),
        "accuracy_aceptadas":
            accuracy_aceptadas
    })


df_aceptadas = (
    pd.DataFrame(
        resultados_aceptadas
    )
    .sort_values("idioma")
)

print("=" * 70)
print("ACCURACY DE PREDICCIONES ACEPTADAS")
print("=" * 70)

display(
    df_aceptadas.round(4)
)

ACCURACY DE PREDICCIONES ACEPTADAS


,idioma,documentos,aceptadas,correctas_aceptadas,accuracy_aceptadas
0,en,20,16,16,1.0
1,es,20,11,11,1.0
2,es_en,20,13,13,1.0
3,ru,20,0,0,NaN


In [25]:
# ============================================================
# CELDA 18 — CAPTURA DE ERRORES POR IDIOMA
# ============================================================

captura = []

for idioma, grupo in df_resultados.groupby("idioma"):

    errores_idioma = grupo[
        ~grupo["correcta"]
    ]

    total_errores = len(
        errores_idioma
    )

    if total_errores > 0:

        capturados = (
            errores_idioma["estado"]
            .isin([
                "revision",
                "rechazada"
            ])
            .sum()
        )

        tasa_captura = (
            capturados /
            total_errores
        )

    else:
        capturados = 0
        tasa_captura = 1.0

    captura.append({
        "idioma": idioma,
        "errores": total_errores,
        "errores_capturados":
            int(capturados),
        "tasa_captura_errores":
            tasa_captura
    })


df_captura = pd.DataFrame(
    captura
)

print("=" * 70)
print("CAPTURA DE ERRORES")
print("=" * 70)

display(
    df_captura.round(4)
)

CAPTURA DE ERRORES


,idioma,errores,errores_capturados,tasa_captura_errores
0,en,1,1,1.0
1,es,5,5,1.0
2,es_en,1,1,1.0
3,ru,4,4,1.0


In [26]:
# ============================================================
# CELDA 19 — COBERTURA WORD VS CHAR POR IDIOMA
# ============================================================

cobertura_idioma = (
    df_resultados
    .groupby("idioma")
    .agg(
        documentos=(
            "case_id",
            "size"
        ),

        word_promedio=(
            "word_features_activas",
            "mean"
        ),

        char_promedio=(
            "char_features_activas",
            "mean"
        ),

        total_promedio=(
            "features_activas_total",
            "mean"
        ),

        margen_promedio=(
            "margen_decision",
            "mean"
        )
    )
    .reset_index()
)

cobertura_idioma["pct_word"] = (
    cobertura_idioma["word_promedio"]
    /
    cobertura_idioma["total_promedio"]
    * 100
)

cobertura_idioma["pct_char"] = (
    cobertura_idioma["char_promedio"]
    /
    cobertura_idioma["total_promedio"]
    * 100
)

print("=" * 70)
print("COBERTURA WORD VS CHAR")
print("=" * 70)

display(
    cobertura_idioma.round(2)
)

COBERTURA WORD VS CHAR


,idioma,documentos,word_promedio,char_promedio,total_promedio,margen_promedio,pct_word,pct_char
0,en,20,14.45,267.70,282.15,1.40,5.12,94.88
1,es,20,2.55,230.25,232.80,0.88,1.10,98.90
2,es_en,20,6.90,246.95,253.85,1.28,2.72,97.28
3,ru,20,3.95,82.90,86.85,1.02,4.55,95.45


In [27]:
# ============================================================
# CELDA 20 — DOCUMENTOS SIN COBERTURA
# ============================================================

sin_cobertura = (
    df_resultados[
        df_resultados[
            "features_activas_total"
        ] == 0
    ]
)

print("=" * 70)
print("DOCUMENTOS SIN COBERTURA")
print("=" * 70)

print(
    "Total:",
    len(sin_cobertura)
)

if len(sin_cobertura) > 0:

    display(
        sin_cobertura[
            [
                "case_id",
                "idioma",
                "categoria_real",
                "texto"
            ]
        ]
    )

else:

    print(
        "\n✅ Todos los documentos "
        "tuvieron cobertura."
    )

DOCUMENTOS SIN COBERTURA
Total: 0

✅ Todos los documentos tuvieron cobertura.


In [28]:
# ============================================================
# CELDA 21 — CONSISTENCIA ENTRE IDIOMAS
# ============================================================

consistencia = (
    df_resultados
    .pivot_table(
        index=[
            "case_id",
            "categoria_real"
        ],
        columns="idioma",
        values="categoria_predicha",
        aggfunc="first"
    )
    .reset_index()
)

# Orden visual
orden_idiomas = [
    "es",
    "en",
    "ru",
    "es_en"
]

columnas_presentes = [
    idioma
    for idioma in orden_idiomas
    if idioma in consistencia.columns
]


consistencia["num_predicciones_distintas"] = (
    consistencia[
        columnas_presentes
    ]
    .nunique(axis=1)
)

consistencia["consistente"] = (
    consistencia[
        "num_predicciones_distintas"
    ] == 1
)


print("=" * 70)
print("CONSISTENCIA CROSS-LANGUAGE")
print("=" * 70)

display(consistencia)

CONSISTENCIA CROSS-LANGUAGE


idioma,case_id,categoria_real,en,es,es_en,ru,num_predicciones_distintas,consistente
0,backend_001,backend,backend,backend,backend,backend,1,True
1,backend_002,backend,backend,backend,backend,backend,1,True
2,backend_003,backend,backend,backend,backend,backend,1,True
3,backend_004,backend,backend,backend,backend,backend,1,True
4,backend_005,backend,backend,backend,backend,backend,1,True
5,cloud_001,cloud,cloud,backend,cloud,cloud,2,False
6,cloud_002,cloud,cloud,cloud,cloud,cloud,1,True
7,cloud_003,cloud,cloud,cloud,cloud,cloud,1,True
8,cloud_004,cloud,cloud,cloud,cloud,cloud,1,True
9,cloud_005,cloud,frontend,frontend,frontend,cloud,2,False


In [29]:
# ============================================================
# CELDA 22 — TASA DE CONSISTENCIA SEMÁNTICA
# ============================================================

total_casos = len(consistencia)

casos_consistentes = int(
    consistencia["consistente"].sum()
)

casos_inconsistentes = (
    total_casos -
    casos_consistentes
)

tasa_consistencia = (
    casos_consistentes /
    total_casos
)

print("=" * 70)
print("CONSISTENCIA ENTRE TRADUCCIONES")
print("=" * 70)

print(f"\nCasos semánticos:    {total_casos}")

print(
    f"Consistentes:       "
    f"{casos_consistentes}"
)

print(
    f"Inconsistentes:     "
    f"{casos_inconsistentes}"
)

print(
    f"Tasa consistencia:  "
    f"{tasa_consistencia:.2%}"
)

CONSISTENCIA ENTRE TRADUCCIONES

Casos semánticos:    20
Consistentes:       13
Inconsistentes:     7
Tasa consistencia:  65.00%


In [30]:
# ============================================================
# CELDA 23 — CASOS SENSIBLES AL IDIOMA
# ============================================================

casos_sensibles_idioma = (
    consistencia[
        ~consistencia["consistente"]
    ]
    .copy()
)

print("=" * 70)
print("CASOS CON CAMBIO DE PREDICCIÓN SEGÚN IDIOMA")
print("=" * 70)

print(
    "\nTotal:",
    len(casos_sensibles_idioma)
)

display(
    casos_sensibles_idioma
)

CASOS CON CAMBIO DE PREDICCIÓN SEGÚN IDIOMA

Total: 7


idioma,case_id,categoria_real,en,es,es_en,ru,num_predicciones_distintas,consistente
5,cloud_001,cloud,cloud,backend,cloud,cloud,2,False
9,cloud_005,cloud,frontend,frontend,frontend,cloud,2,False
11,datascience_002,datascience,datascience,datascience,datascience,cloud,2,False
12,datascience_003,datascience,datascience,cloud,datascience,cloud,2,False
13,datascience_004,datascience,datascience,datascience,datascience,cloud,2,False
14,datascience_005,datascience,datascience,backend,datascience,cloud,3,False
19,frontend_005,frontend,frontend,backend,frontend,frontend,2,False


In [31]:
# ============================================================
# CELDA 24 — CLASIFICACIÓN DE ESTABILIDAD CROSS-LANGUAGE
# ============================================================

def clasificar_caso(row):

    predicciones = [
        row["es"],
        row["en"],
        row["ru"],
        row["es_en"]
    ]

    real = row["categoria_real"]

    # Todas iguales
    if len(set(predicciones)) == 1:

        if predicciones[0] == real:
            return "estable_correcto"

        return "estable_incorrecto"

    # Cambia según idioma
    return "sensible_idioma"


consistencia["tipo_estabilidad"] = (
    consistencia.apply(
        clasificar_caso,
        axis=1
    )
)


print("=" * 70)
print("CLASIFICACIÓN DE ESTABILIDAD")
print("=" * 70)

resumen_estabilidad = (
    consistencia[
        "tipo_estabilidad"
    ]
    .value_counts()
)

display(
    resumen_estabilidad
)

print()

display(
    consistencia[
        [
            "case_id",
            "categoria_real",
            "es",
            "en",
            "ru",
            "es_en",
            "tipo_estabilidad"
        ]
    ]
)

CLASIFICACIÓN DE ESTABILIDAD


tipo_estabilidad
estable_correcto    13
sensible_idioma      7
Name: count, dtype: int64

idioma,case_id,categoria_real,es,en,ru,es_en,tipo_estabilidad
0,backend_001,backend,backend,backend,backend,backend,estable_correcto
1,backend_002,backend,backend,backend,backend,backend,estable_correcto
2,backend_003,backend,backend,backend,backend,backend,estable_correcto
3,backend_004,backend,backend,backend,backend,backend,estable_correcto
4,backend_005,backend,backend,backend,backend,backend,estable_correcto
5,cloud_001,cloud,backend,cloud,cloud,cloud,sensible_idioma
6,cloud_002,cloud,cloud,cloud,cloud,cloud,estable_correcto
7,cloud_003,cloud,cloud,cloud,cloud,cloud,estable_correcto
8,cloud_004,cloud,cloud,cloud,cloud,cloud,estable_correcto
9,cloud_005,cloud,frontend,frontend,cloud,frontend,sensible_idioma


In [32]:
# ============================================================
# CELDA 25 — CONSISTENCIA POR CATEGORÍA
# ============================================================

consistencia_categoria = (
    consistencia
    .groupby("categoria_real")
    .agg(
        casos=(
            "case_id",
            "count"
        ),

        consistentes=(
            "consistente",
            "sum"
        ),

        tasa_consistencia=(
            "consistente",
            "mean"
        )
    )
    .reset_index()
)

print("=" * 70)
print("CONSISTENCIA POR CATEGORÍA")
print("=" * 70)

display(
    consistencia_categoria.round(4)
)

CONSISTENCIA POR CATEGORÍA


,categoria_real,casos,consistentes,tasa_consistencia
0,backend,5,5,1.0
1,cloud,5,3,0.6
2,datascience,5,1,0.2
3,frontend,5,4,0.8


In [33]:
# ============================================================
# CELDA 26 — ACUERDO ENTRE PARES DE IDIOMAS
# ============================================================

idiomas = [
    "es",
    "en",
    "ru",
    "es_en"
]

matriz_acuerdo = pd.DataFrame(
    index=idiomas,
    columns=idiomas,
    dtype=float
)

for idioma_a in idiomas:

    for idioma_b in idiomas:

        matriz_acuerdo.loc[
            idioma_a,
            idioma_b
        ] = (
            consistencia[
                idioma_a
            ]
            ==
            consistencia[
                idioma_b
            ]
        ).mean()


print("=" * 70)
print("ACUERDO ENTRE IDIOMAS")
print("=" * 70)

display(
    matriz_acuerdo.round(2)
)

ACUERDO ENTRE IDIOMAS


,es,en,ru,es_en
es,1.0,0.80,0.70,0.80
en,0.8,1.00,0.75,1.00
ru,0.7,0.75,1.00,0.75
es_en,0.8,1.00,0.75,1.00


In [34]:
# ============================================================
# CELDA 27 — DATASET DE CASOS SENSIBLES
# ============================================================

ids_sensibles = (
    casos_sensibles_idioma[
        "case_id"
    ]
    .tolist()
)

df_sensibles = (
    df_resultados[
        df_resultados[
            "case_id"
        ].isin(ids_sensibles)
    ]
    .copy()
)

df_sensibles = (
    df_sensibles
    .sort_values(
        [
            "case_id",
            "idioma"
        ]
    )
)

print("=" * 70)
print("CASOS SENSIBLES AL IDIOMA")
print("=" * 70)

print(
    f"\nCasos semánticos: "
    f"{len(ids_sensibles)}"
)

print(
    f"Textos a analizar: "
    f"{len(df_sensibles)}"
)

display(
    df_sensibles[
        [
            "case_id",
            "idioma",
            "categoria_real",
            "categoria_predicha",
            "estado",
            "margen_decision",
            "features_activas_total"
        ]
    ]
)

CASOS SENSIBLES AL IDIOMA

Casos semánticos: 7
Textos a analizar: 28


,case_id,idioma,categoria_real,categoria_predicha,estado,margen_decision,features_activas_total
21,cloud_001,en,cloud,cloud,aceptada,1.820642,232
20,cloud_001,es,cloud,backend,revision,0.315313,234
23,cloud_001,es_en,cloud,cloud,revision,0.227955,189
22,cloud_001,ru,cloud,cloud,revision,0.829652,125
37,cloud_005,en,cloud,frontend,revision,0.304943,340
36,cloud_005,es,cloud,frontend,revision,0.156966,289
39,cloud_005,es_en,cloud,frontend,revision,0.059941,293
38,cloud_005,ru,cloud,cloud,revision,0.241159,135
45,datascience_002,en,datascience,datascience,aceptada,1.624340,284
44,datascience_002,es,datascience,datascience,aceptada,0.893324,229


In [37]:
# ============================================================
# CELDA 28 — EXPLICABILIDAD DE CASOS SENSIBLES
# ============================================================

print("=" * 70)
print("EXPLICABILIDAD — CASOS SENSIBLES")
print("=" * 70)

textos_sensibles = (
    df_sensibles["texto"]
    .astype(str)
    .tolist()
)

print(
    "\nTextos a explicar:",
    len(textos_sensibles)
)

resultado_explicable = predictor.predict(
    textos_sensibles,
    include_explanation=True,
    explanation_top_n=10,
    top_k=4
)

print(
    "\nPredicciones explicadas:",
    len(resultado_explicable["resultados"])
)

print("\n✅ Explicabilidad completada")

EXPLICABILIDAD — CASOS SENSIBLES

Textos a explicar: 28

Predicciones explicadas: 28

✅ Explicabilidad completada


In [38]:
# ============================================================
# CELDA 29 — INSPECCIONAR CONTRATO DE EXPLICABILIDAD
# ============================================================

primera_prediccion = (
    resultado_explicable["resultados"][0]
)

print("=" * 70)
print("CAMPOS DE UNA PREDICCIÓN EXPLICABLE")
print("=" * 70)

for clave in primera_prediccion.keys():
    print(f"- {clave}")


print("\n" + "=" * 70)
print("ESTRUCTURA DE LA EXPLICACIÓN")
print("=" * 70)

explicacion = primera_prediccion.get(
    "explicacion"
)

if explicacion is None:
    print(
        "\n⚠️ No existe una clave llamada "
        "'explicacion'."
    )

    print(
        "\nBuscando campos relacionados..."
    )

    for clave, valor in primera_prediccion.items():

        nombre = clave.lower()

        if (
            "explic" in nombre
            or "term" in nombre
            or "feature" in nombre
            or "contrib" in nombre
        ):
            print(f"\n--- {clave} ---")

            try:
                print(
                    json.dumps(
                        valor,
                        ensure_ascii=False,
                        indent=2
                    )
                )
            except TypeError:
                print(valor)

else:

    print(
        json.dumps(
            explicacion,
            ensure_ascii=False,
            indent=2
        )
    )

CAMPOS DE UNA PREDICCIÓN EXPLICABLE
- request_id
- record_id
- timestamp_utc
- input_type
- text
- characters
- words
- valid_input
- validation_message
- estado
- categoria_predicha
- segunda_categoria
- puntuacion_ganadora
- puntuacion_segunda
- margen_decision
- nivel_margen
- terminos_activos
- word_features_activas
- char_features_activas
- features_activas_total
- accion_recomendada
- requiere_revision
- prediccion_utilizable
- advertencias
- ranking_categorias
- explicacion

ESTRUCTURA DE LA EXPLICACIÓN
{
  "positive_terms": [
    {
      "term": "increase",
      "feature_type": "word",
      "tfidf": 0.37179940528849587,
      "coefficient": 0.3500555469150029,
      "contribution": 0.13015044416093724
    },
    {
      "term": "capacity",
      "feature_type": "word",
      "tfidf": 0.32142479281072067,
      "coefficient": 0.32692537588430165,
      "contribution": 0.10508192120817864
    },
    {
      "term": "using",
      "feature_type": "word",
      "tfidf": 0.1996941

In [39]:
# ============================================================
# CELDA 30 — CONSOLIDAR RESULTADOS EXPLICABLES
# ============================================================

df_explicable = pd.DataFrame(
    resultado_explicable["resultados"]
)

df_analisis = pd.concat(
    [
        df_sensibles.reset_index(drop=True)[
            [
                "case_id",
                "idioma",
                "categoria_real",
                "texto"
            ]
        ],
        df_explicable.reset_index(drop=True)
    ],
    axis=1
)

df_analisis["correcta"] = (
    df_analisis["categoria_real"]
    ==
    df_analisis["categoria_predicha"]
)

print("=" * 70)
print("DATASET EXPLICABLE")
print("=" * 70)

print("\nFilas:", len(df_analisis))

print(
    "Correctas:",
    int(df_analisis["correcta"].sum())
)

print(
    "Incorrectas:",
    int((~df_analisis["correcta"]).sum())
)

display(
    df_analisis[
        [
            "case_id",
            "idioma",
            "categoria_real",
            "categoria_predicha",
            "segunda_categoria",
            "estado",
            "margen_decision",
            "word_features_activas",
            "char_features_activas",
            "features_activas_total",
            "correcta"
        ]
    ]
)

DATASET EXPLICABLE

Filas: 28
Correctas: 17
Incorrectas: 11


,case_id,idioma,categoria_real,categoria_predicha,segunda_categoria,estado,margen_decision,word_features_activas,char_features_activas,features_activas_total,correcta
0,cloud_001,en,cloud,cloud,backend,aceptada,1.820642,14,218,232,True
1,cloud_001,es,cloud,backend,cloud,revision,0.315313,5,229,234,False
2,cloud_001,es_en,cloud,cloud,backend,revision,0.227955,6,183,189,True
3,cloud_001,ru,cloud,cloud,backend,revision,0.829652,6,119,125,True
4,cloud_005,en,cloud,frontend,cloud,revision,0.304943,19,321,340,False
5,cloud_005,es,cloud,frontend,cloud,revision,0.156966,4,285,289,False
6,cloud_005,es_en,cloud,frontend,cloud,revision,0.059941,9,284,293,False
7,cloud_005,ru,cloud,cloud,datascience,revision,0.241159,7,128,135,True
8,datascience_002,en,datascience,datascience,frontend,aceptada,1.624340,14,270,284,True
9,datascience_002,es,datascience,datascience,frontend,aceptada,0.893324,3,226,229,True


In [40]:
# ============================================================
# CELDA 31 — VISUALIZADOR DE EXPLICABILIDAD
# ============================================================

def mostrar_explicacion_caso(
    df,
    case_id,
    idioma=None,
    top_n=10
):

    datos = df[
        df["case_id"] == case_id
    ]

    if idioma is not None:
        datos = datos[
            datos["idioma"] == idioma
        ]

    if datos.empty:
        print(
            f"No se encontró el caso {case_id}"
        )
        return

    orden_idiomas = {
        "es": 0,
        "en": 1,
        "es_en": 2,
        "ru": 3
    }

    datos = datos.copy()

    datos["_orden"] = (
        datos["idioma"]
        .map(orden_idiomas)
        .fillna(99)
    )

    datos = datos.sort_values("_orden")


    for _, row in datos.iterrows():

        print("\n" + "=" * 85)

        print(
            f"CASO: {row['case_id']} "
            f"| IDIOMA: {row['idioma']}"
        )

        print("=" * 85)

        print(
            f"Real:        "
            f"{row['categoria_real']}"
        )

        print(
            f"Predicción:  "
            f"{row['categoria_predicha']}"
        )

        print(
            f"2ª categoría:"
            f" {row['segunda_categoria']}"
        )

        print(
            f"Correcta:    "
            f"{row['correcta']}"
        )

        print(
            f"Estado:      "
            f"{row['estado']}"
        )

        print(
            f"Margen:      "
            f"{float(row['margen_decision']):.4f}"
        )

        print(
            "\nCobertura:"
        )

        print(
            f"  Word:  "
            f"{row['word_features_activas']}"
        )

        print(
            f"  Char:  "
            f"{row['char_features_activas']}"
        )

        print(
            f"  Total: "
            f"{row['features_activas_total']}"
        )

        print("\nTexto:")
        print(row["texto"])


        exp = row.get("explicacion")

        if not isinstance(exp, dict):
            print(
                "\n⚠ Sin explicación estructurada"
            )
            continue


        # ----------------------------------------------------
        # DIFFERENTIAL TERMS
        # ----------------------------------------------------

        print(
            "\n--- FEATURES DIFERENCIALES ---"
        )

        differential = exp.get(
            "differential_terms",
            []
        )

        if not differential:
            print("Sin datos")

        for item in differential[:top_n]:

            print(
                f"{item['term']!r:<24} "
                f"{item['feature_type']:<6} "
                f"contrib={item['contribution']:+.4f} "
                f"→ {item['favours']}"
            )


        # ----------------------------------------------------
        # POSITIVE TERMS
        # ----------------------------------------------------

        print(
            "\n--- FEATURES POSITIVAS ---"
        )

        positive = exp.get(
            "positive_terms",
            []
        )

        for item in positive[:top_n]:

            print(
                f"{item['term']!r:<24} "
                f"{item['feature_type']:<6} "
                f"tfidf={item['tfidf']:.4f} "
                f"coef={item['coefficient']:+.4f} "
                f"contrib={item['contribution']:+.4f}"
            )

In [41]:
# ============================================================
# CELDA 32 — CASO MÁS INESTABLE
# ============================================================

mostrar_explicacion_caso(
    df_analisis,
    case_id="datascience_005",
    top_n=10
)


CASO: datascience_005 | IDIOMA: es
Real:        datascience
Predicción:  backend
2ª categoría: cloud
Correcta:    False
Estado:      revision
Margen:      0.3439

Cobertura:
  Word:  5
  Char:  258
  Total: 263

Texto:
Reducir variables con PCA, aplicar K-Means para segmentar observaciones y usar silhouette score para seleccionar el número de clusters.

--- FEATURES DIFERENCIALES ---
'numero'                 word   contrib=+0.2273 → backend
'score'                  word   contrib=-0.1694 → cloud
'usar'                   word   contrib=+0.1604 → backend
'clusters'               word   contrib=-0.1568 → cloud
'variables'              word   contrib=-0.0802 → cloud
'ar '                    char   contrib=+0.0461 → backend
' cl'                    char   contrib=-0.0456 → cloud
'ra '                    char   contrib=-0.0260 → cloud
'cion'                   char   contrib=+0.0257 → backend
' par'                   char   contrib=+0.0249 → backend

--- FEATURES POSITIVAS ---
'numero'      

In [42]:
# ============================================================
# CELDA 33 — EXTRAER FEATURES DIFERENCIALES
# ============================================================

filas_diferenciales = []

for _, row in df_analisis.iterrows():

    exp = row.get(
        "explicacion"
    )

    if not isinstance(
        exp,
        dict
    ):
        continue

    for posicion, item in enumerate(
        exp.get(
            "differential_terms",
            []
        ),
        start=1
    ):

        filas_diferenciales.append({

            "case_id":
                row["case_id"],

            "idioma":
                row["idioma"],

            "categoria_real":
                row["categoria_real"],

            "categoria_predicha":
                row["categoria_predicha"],

            "segunda_categoria":
                row["segunda_categoria"],

            "correcta":
                row["correcta"],

            "posicion":
                posicion,

            "term":
                item["term"],

            "feature_type":
                item["feature_type"],

            "contribution":
                item["contribution"],

            "favours":
                item["favours"]
        })


df_differential = pd.DataFrame(
    filas_diferenciales
)


print("=" * 70)
print("FEATURES DIFERENCIALES")
print("=" * 70)

print(
    "\nTotal:",
    len(df_differential)
)

display(
    df_differential.head(20)
)

FEATURES DIFERENCIALES

Total: 280


,case_id,idioma,categoria_real,categoria_predicha,segunda_categoria,correcta,posicion,term,feature_type,contribution,favours
0,cloud_001,en,cloud,cloud,backend,True,1,capacity,word,0.163705,cloud
1,cloud_001,en,cloud,cloud,backend,True,2,using,word,0.159987,cloud
2,cloud_001,en,cloud,cloud,backend,True,3,increase,word,0.155818,cloud
3,cloud_001,en,cloud,cloud,backend,True,4,an,word,0.110418,cloud
4,cloud_001,en,cloud,cloud,backend,True,5,aws,char,0.088741,cloud
5,cloud_001,en,cloud,cloud,backend,True,6,aws,char,0.088005,cloud
6,cloud_001,en,cloud,cloud,backend,True,7,aws,char,0.085535,cloud
7,cloud_001,en,cloud,cloud,backend,True,8,aws,char,0.084767,cloud
8,cloud_001,en,cloud,cloud,backend,True,9,aw,char,0.081370,cloud
9,cloud_001,en,cloud,cloud,backend,True,10,scaling,word,-0.078383,backend


In [43]:
# ============================================================
# CELDA 34 — WORD VS CHAR EN FEATURES DIFERENCIALES
# ============================================================

tabla_tipo_diferencial = (
    pd.crosstab(
        df_differential["idioma"],
        df_differential["feature_type"]
    )
)

print("=" * 70)
print("WORD VS CHAR — FEATURES DIFERENCIALES")
print("=" * 70)

display(
    tabla_tipo_diferencial
)

WORD VS CHAR — FEATURES DIFERENCIALES


feature_type,char,word
idioma,,
en,17,53
es,50,20
es_en,29,41
ru,46,24


In [44]:
tabla_tipo_diferencial_pct = (
    pd.crosstab(
        df_differential["idioma"],
        df_differential["feature_type"],
        normalize="index"
    )
    * 100
)

display(
    tabla_tipo_diferencial_pct.round(2)
)

feature_type,char,word
idioma,,
en,24.29,75.71
es,71.43,28.57
es_en,41.43,58.57
ru,65.71,34.29


In [45]:
# ============================================================
# CELDA 35 — MAGNITUD DE CONTRIBUCIÓN WORD VS CHAR
# ============================================================

df_differential[
    "contribution_abs"
] = (
    df_differential[
        "contribution"
    ].abs()
)

contribucion_tipo = (
    df_differential
    .groupby(
        [
            "idioma",
            "feature_type"
        ]
    )
    ["contribution_abs"]
    .sum()
    .unstack(
        fill_value=0
    )
)

contribucion_pct = (
    contribucion_tipo
    .div(
        contribucion_tipo.sum(
            axis=1
        ),
        axis=0
    )
    * 100
)

print("=" * 70)
print(
    "CONTRIBUCIÓN DIFERENCIAL WORD VS CHAR"
)
print("=" * 70)

print("\nMagnitud acumulada:")

display(
    contribucion_tipo.round(4)
)

print(
    "\nPorcentaje de contribución:"
)

display(
    contribucion_pct.round(2)
)

CONTRIBUCIÓN DIFERENCIAL WORD VS CHAR

Magnitud acumulada:


feature_type,char,word
idioma,,
en,0.9100,5.3691
es,2.0825,2.8921
es_en,1.3920,4.9805
ru,3.7318,2.9747



Porcentaje de contribución:


feature_type,char,word
idioma,,
en,14.49,85.51
es,41.86,58.14
es_en,21.84,78.16
ru,55.64,44.36


In [46]:
# ============================================================
# CELDA 36 — CONTRIBUCIÓN SEGÚN ACIERTO / ERROR
# ============================================================

contribucion_correcta = (
    df_differential
    .groupby(
        [
            "correcta",
            "feature_type"
        ]
    )
    ["contribution_abs"]
    .sum()
    .unstack(
        fill_value=0
    )
)

contribucion_correcta_pct = (
    contribucion_correcta
    .div(
        contribucion_correcta.sum(
            axis=1
        ),
        axis=0
    )
    * 100
)

print("=" * 70)
print("CONTRIBUCIÓN EN ACIERTOS VS ERRORES")
print("=" * 70)

display(
    contribucion_correcta_pct.round(2)
)

CONTRIBUCIÓN EN ACIERTOS VS ERRORES


feature_type,char,word
correcta,,
False,41.03,58.97
True,28.74,71.26


In [47]:
# ============================================================
# CELDA 37 — WORD FEATURES MÁS FRECUENTES
# ============================================================

word_differential = (
    df_differential[
        df_differential[
            "feature_type"
        ] == "word"
    ]
)

resumen_word = (
    word_differential
    .groupby("term")
    .agg(
        apariciones=(
            "term",
            "size"
        ),

        contribucion_media=(
            "contribution_abs",
            "mean"
        ),

        contribucion_total=(
            "contribution_abs",
            "sum"
        )
    )
    .sort_values(
        [
            "apariciones",
            "contribucion_total"
        ],
        ascending=False
    )
)

print("=" * 70)
print(
    "WORD FEATURES DIFERENCIALES MÁS FRECUENTES"
)
print("=" * 70)

display(
    resumen_word.head(30).round(4)
)

WORD FEATURES DIFERENCIALES MÁS FRECUENTES


,apariciones,contribucion_media,contribucion_total
term,,,
и,6,0.1134,0.6803
rolling,4,0.2022,0.8088
fetch,4,0.1929,0.7717
для,4,0.1606,0.6423
rollback,4,0.1138,0.4550
score,4,0.1021,0.4086
scaling,4,0.0938,0.3752
images,4,0.0937,0.3747
recall,4,0.0861,0.3446


In [48]:
# ============================================================
# CELDA 34 — WORD VS CHAR EN FEATURES DIFERENCIALES
# ============================================================

tabla_tipo_diferencial = pd.crosstab(
    df_differential["idioma"],
    df_differential["feature_type"]
)

tabla_tipo_diferencial_pct = (
    pd.crosstab(
        df_differential["idioma"],
        df_differential["feature_type"],
        normalize="index"
    )
    * 100
)

print("=" * 70)
print("WORD VS CHAR — FEATURES DIFERENCIALES")
print("=" * 70)

print("\nCantidad:")
display(tabla_tipo_diferencial)

print("\nPorcentaje:")
display(
    tabla_tipo_diferencial_pct.round(2)
)

WORD VS CHAR — FEATURES DIFERENCIALES

Cantidad:


feature_type,char,word
idioma,,
en,17,53
es,50,20
es_en,29,41
ru,46,24



Porcentaje:


feature_type,char,word
idioma,,
en,24.29,75.71
es,71.43,28.57
es_en,41.43,58.57
ru,65.71,34.29


In [49]:
# ============================================================
# CELDA 35 — CONTRIBUCIÓN WORD VS CHAR
# ============================================================

df_differential["contribution_abs"] = (
    df_differential["contribution"].abs()
)

contribucion_tipo = (
    df_differential
    .groupby([
        "idioma",
        "feature_type"
    ])["contribution_abs"]
    .sum()
    .unstack(fill_value=0)
)

contribucion_pct = (
    contribucion_tipo
    .div(
        contribucion_tipo.sum(axis=1),
        axis=0
    )
    * 100
)

print("=" * 70)
print("MAGNITUD DE CONTRIBUCIÓN WORD VS CHAR")
print("=" * 70)

print("\nContribución absoluta acumulada:")
display(
    contribucion_tipo.round(4)
)

print("\nParticipación porcentual:")
display(
    contribucion_pct.round(2)
)

MAGNITUD DE CONTRIBUCIÓN WORD VS CHAR

Contribución absoluta acumulada:


feature_type,char,word
idioma,,
en,0.9100,5.3691
es,2.0825,2.8921
es_en,1.3920,4.9805
ru,3.7318,2.9747



Participación porcentual:


feature_type,char,word
idioma,,
en,14.49,85.51
es,41.86,58.14
es_en,21.84,78.16
ru,55.64,44.36


In [50]:
# ============================================================
# CELDA 36 — WORD/CHAR EN ACIERTOS VS ERRORES
# ============================================================

contribucion_acierto = (
    df_differential
    .groupby([
        "correcta",
        "feature_type"
    ])["contribution_abs"]
    .sum()
    .unstack(fill_value=0)
)

contribucion_acierto_pct = (
    contribucion_acierto
    .div(
        contribucion_acierto.sum(axis=1),
        axis=0
    )
    * 100
)

print("=" * 70)
print("CONTRIBUCIÓN — ACIERTOS VS ERRORES")
print("=" * 70)

display(
    contribucion_acierto_pct.round(2)
)

CONTRIBUCIÓN — ACIERTOS VS ERRORES


feature_type,char,word
correcta,,
False,41.03,58.97
True,28.74,71.26


In [51]:
# ============================================================
# CELDA 37 — WORD FEATURES MÁS INFLUYENTES
# ============================================================

df_word = (
    df_differential[
        df_differential["feature_type"] == "word"
    ]
    .copy()
)

resumen_word = (
    df_word
    .groupby("term")
    .agg(
        apariciones=("term", "size"),
        contribucion_media=(
            "contribution_abs",
            "mean"
        ),
        contribucion_total=(
            "contribution_abs",
            "sum"
        )
    )
    .sort_values(
        [
            "contribucion_total",
            "apariciones"
        ],
        ascending=False
    )
)

print("=" * 70)
print("WORD FEATURES MÁS INFLUYENTES")
print("=" * 70)

display(
    resumen_word.head(30).round(4)
)

WORD FEATURES MÁS INFLUYENTES


,apariciones,contribucion_media,contribucion_total
term,,,
rolling,4,0.2022,0.8088
fetch,4,0.1929,0.7717
и,6,0.1134,0.6803
для,4,0.1606,0.6423
compare,3,0.1956,0.5867
loss,2,0.2864,0.5727
loading,2,0.2439,0.4877
validation,3,0.1576,0.4728
rollback,4,0.1138,0.4550


In [52]:
# ============================================================
# CELDA 38 — WORD FEATURES EN ACIERTOS VS ERRORES
# ============================================================

resumen_word_acierto = (
    df_word
    .groupby(
        [
            "correcta",
            "term"
        ]
    )
    .agg(
        apariciones=(
            "term",
            "size"
        ),

        contribucion_media=(
            "contribution_abs",
            "mean"
        ),

        contribucion_total=(
            "contribution_abs",
            "sum"
        )
    )
    .reset_index()
)


print("=" * 70)
print("WORD FEATURES — PREDICCIONES INCORRECTAS")
print("=" * 70)

display(
    resumen_word_acierto[
        resumen_word_acierto["correcta"] == False
    ]
    .sort_values(
        "contribucion_total",
        ascending=False
    )
    .head(30)
    .round(4)
)


print("\n" + "=" * 70)
print("WORD FEATURES — PREDICCIONES CORRECTAS")
print("=" * 70)

display(
    resumen_word_acierto[
        resumen_word_acierto["correcta"] == True
    ]
    .sort_values(
        "contribucion_total",
        ascending=False
    )
    .head(30)
    .round(4)
)

WORD FEATURES — PREDICCIONES INCORRECTAS


,correcta,term,apariciones,contribucion_media,contribucion_total
17,False,rolling,3,0.2297,0.6892
28,False,для,3,0.1651,0.4953
7,False,imagenes,2,0.2067,0.4135
16,False,rollback,3,0.1197,0.3592
29,False,и,4,0.0732,0.2928
6,False,fetch,1,0.2720,0.2720
19,False,score,2,0.1310,0.2620
25,False,validation,1,0.2418,0.2418
8,False,images,2,0.1195,0.2390
20,False,training,1,0.2349,0.2349



WORD FEATURES — PREDICCIONES CORRECTAS


,correcta,term,apariciones,contribucion_media,contribucion_total
37,True,compare,3,0.1956,0.5867
53,True,loss,2,0.2864,0.5727
46,True,fetch,3,0.1666,0.4997
50,True,loading,2,0.2439,0.4877
41,True,events,2,0.2004,0.4007
89,True,и,2,0.1937,0.3874
85,True,using,2,0.1879,0.3758
71,True,series,2,0.1711,0.3422
54,True,models,2,0.1631,0.3262
30,True,an,3,0.1006,0.3018


In [53]:
# ============================================================
# CELDA 39 — FEATURES QUE FAVORECEN ERRORES
# ============================================================

df_error_features = (
    df_differential[
        df_differential["correcta"] == False
    ]
    .copy()
)

df_error_features["favorece_error"] = (
    df_error_features["favours"]
    ==
    df_error_features["categoria_predicha"]
)


features_error = (
    df_error_features[
        df_error_features[
            "favorece_error"
        ]
    ]
    .groupby([
        "term",
        "feature_type"
    ])
    .agg(
        apariciones=(
            "term",
            "size"
        ),

        contribucion_media=(
            "contribution_abs",
            "mean"
        ),

        contribucion_total=(
            "contribution_abs",
            "sum"
        )
    )
    .sort_values(
        "contribucion_total",
        ascending=False
    )
)


print("=" * 70)
print("FEATURES QUE EMPUJAN HACIA CATEGORÍAS INCORRECTAS")
print("=" * 70)

display(
    features_error
    .head(30)
    .round(4)
)

FEATURES QUE EMPUJAN HACIA CATEGORÍAS INCORRECTAS


,,apariciones,contribucion_media,contribucion_total
term,feature_type,,,
rolling,word,3,0.2297,0.6892
для,word,3,0.1651,0.4953
пр,char,3,0.1609,0.4826
и,char,4,0.0859,0.3438
про,char,1,0.3194,0.3194
images,word,2,0.1195,0.2390
training,word,1,0.2349,0.2349
numero,word,1,0.2273,0.2273
imagenes,word,1,0.2097,0.2097


In [54]:
# ============================================================
# CELDA 40 — FEATURES DE ERROR POR IDIOMA
# ============================================================

resumen_error_idioma = (
    df_error_features[
        df_error_features[
            "favorece_error"
        ]
    ]
    .groupby([
        "idioma",
        "feature_type"
    ])
    ["contribution_abs"]
    .sum()
    .unstack(
        fill_value=0
    )
)

resumen_error_idioma_pct = (
    resumen_error_idioma
    .div(
        resumen_error_idioma.sum(axis=1),
        axis=0
    )
    * 100
)


print("=" * 70)
print("SEÑAL HACIA EL ERROR POR IDIOMA")
print("=" * 70)

display(
    resumen_error_idioma_pct.round(2)
)

SEÑAL HACIA EL ERROR POR IDIOMA


feature_type,char,word
idioma,,
en,0.00,100.00
es,39.53,60.47
es_en,0.00,100.00
ru,62.21,37.79


In [55]:
# ============================================================
# CELDA 41 — MATRIZ DE CONFUSIONES MULTILINGÜES
# ============================================================

tabla_confusiones = (
    df_resultados[
        ~df_resultados["correcta"]
    ]
    .groupby([
        "idioma",
        "categoria_real",
        "categoria_predicha"
    ])
    .size()
    .reset_index(
        name="casos"
    )
    .sort_values(
        [
            "idioma",
            "casos"
        ],
        ascending=[
            True,
            False
        ]
    )
)

print("=" * 70)
print("PATRONES DE CONFUSIÓN")
print("=" * 70)

display(
    tabla_confusiones
)

PATRONES DE CONFUSIÓN


,idioma,categoria_real,categoria_predicha,casos
0,en,cloud,frontend,1
1,es,cloud,backend,1
2,es,cloud,frontend,1
3,es,datascience,backend,1
4,es,datascience,cloud,1
5,es,frontend,backend,1
6,es_en,cloud,frontend,1
7,ru,datascience,cloud,4


In [56]:
# ============================================================
# CELDA 42 — TAXONOMÍA DE ERRORES MULTILINGÜES
# ============================================================

def clasificar_tipo_error(row):

    idioma = row["idioma"]

    if idioma == "ru":
        return "linguistic_OOD"

    if row["case_id"] == "cloud_005":
        return "semantic_boundary"

    return "lexical_sensitivity"


df_errores_taxonomia = (
    df_resultados[
        ~df_resultados["correcta"]
    ]
    .copy()
)

df_errores_taxonomia[
    "failure_type"
] = df_errores_taxonomia.apply(
    clasificar_tipo_error,
    axis=1
)


print("=" * 70)
print("TAXONOMÍA DE ERRORES")
print("=" * 70)

display(
    df_errores_taxonomia[
        [
            "case_id",
            "idioma",
            "categoria_real",
            "categoria_predicha",
            "estado",
            "failure_type"
        ]
    ]
)

TAXONOMÍA DE ERRORES


,case_id,idioma,categoria_real,categoria_predicha,estado,failure_type
20,cloud_001,es,cloud,backend,revision,lexical_sensitivity
36,cloud_005,es,cloud,frontend,revision,semantic_boundary
37,cloud_005,en,cloud,frontend,revision,semantic_boundary
39,cloud_005,es_en,cloud,frontend,revision,semantic_boundary
46,datascience_002,ru,datascience,cloud,revision,linguistic_OOD
48,datascience_003,es,datascience,cloud,revision,lexical_sensitivity
50,datascience_003,ru,datascience,cloud,revision,linguistic_OOD
54,datascience_004,ru,datascience,cloud,revision,linguistic_OOD
56,datascience_005,es,datascience,backend,revision,lexical_sensitivity
58,datascience_005,ru,datascience,cloud,revision,linguistic_OOD


In [57]:
resumen_failure_type = (
    df_errores_taxonomia[
        "failure_type"
    ]
    .value_counts()
    .rename_axis("failure_type")
    .reset_index(name="casos")
)

display(resumen_failure_type)

,failure_type,casos
0,lexical_sensitivity,4
1,linguistic_OOD,4
2,semantic_boundary,3


In [58]:
# ============================================================
# CELDA 43 — CONTRAFACTUAL CLOUD_005
# ============================================================

contrafactual_cloud = pd.DataFrame({
    "variante": [
        "original",
        "sin_rolling",
        "deployment_updates",
        "cloud_deployment"
    ],

    "texto": [

        (
            "Build a deployment pipeline that creates Docker images, "
            "publishes them to a registry, and performs rolling updates "
            "with automatic rollback."
        ),

        (
            "Build a deployment pipeline that creates Docker images, "
            "publishes them to a registry, and performs updates "
            "with automatic rollback."
        ),

        (
            "Build a deployment pipeline that creates Docker images, "
            "publishes them to a registry, and performs deployment updates "
            "with automatic rollback."
        ),

        (
            "Build a cloud deployment pipeline that creates Docker images, "
            "publishes them to a container registry, and performs deployment "
            "updates with automatic rollback."
        )
    ]
})


resultado_cf_cloud = predictor.predict(
    contrafactual_cloud["texto"].tolist(),
    include_explanation=True,
    explanation_top_n=8,
    top_k=4
)

pred_cf_cloud = pd.DataFrame(
    resultado_cf_cloud["resultados"]
)


df_cf_cloud = pd.concat(
    [
        contrafactual_cloud,
        pred_cf_cloud
    ],
    axis=1
)


print("=" * 70)
print("CONTRAFACTUAL — CLOUD_005")
print("=" * 70)

display(
    df_cf_cloud[
        [
            "variante",
            "categoria_predicha",
            "segunda_categoria",
            "estado",
            "margen_decision",
            "word_features_activas",
            "char_features_activas",
            "features_activas_total"
        ]
    ]
)

CONTRAFACTUAL — CLOUD_005


,variante,categoria_predicha,segunda_categoria,estado,margen_decision,word_features_activas,char_features_activas,features_activas_total
0,original,frontend,cloud,revision,0.304943,19,321,340
1,sin_rolling,frontend,cloud,revision,0.168379,18,309,327
2,deployment_updates,frontend,cloud,revision,0.066067,18,309,327
3,cloud_deployment,cloud,frontend,aceptada,1.264184,20,351,371


In [59]:
# ============================================================
# CELDA 44 — CONTRAFACTUAL DATASCIENCE_005 ES
# ============================================================

contrafactual_ds_es = pd.DataFrame({
    "variante": [
        "original",
        "mas_tecnico",
        "terminologia_ml",
        "ingles_tecnico"
    ],

    "texto": [

        (
            "Reducir variables con PCA, aplicar K-Means para segmentar "
            "observaciones y usar silhouette score para seleccionar "
            "el número de clusters."
        ),

        (
            "Aplicar PCA para reducción de dimensionalidad y K-Means "
            "para clustering, evaluando los clusters mediante "
            "silhouette score."
        ),

        (
            "Entrenar un modelo de clustering con K-Means después de "
            "reducir dimensionalidad mediante PCA y evaluar la calidad "
            "de los grupos con silhouette score."
        ),

        (
            "Aplicar PCA para dimensionality reduction y K-Means "
            "para clustering, evaluando los clusters mediante "
            "silhouette score."
        )
    ]
})


resultado_cf_ds = predictor.predict(
    contrafactual_ds_es["texto"].tolist(),
    include_explanation=True,
    explanation_top_n=8,
    top_k=4
)


pred_cf_ds = pd.DataFrame(
    resultado_cf_ds["resultados"]
)


df_cf_ds = pd.concat(
    [
        contrafactual_ds_es,
        pred_cf_ds
    ],
    axis=1
)


print("=" * 70)
print("CONTRAFACTUAL — DATASCIENCE_005 ES")
print("=" * 70)

display(
    df_cf_ds[
        [
            "variante",
            "categoria_predicha",
            "segunda_categoria",
            "estado",
            "margen_decision",
            "word_features_activas",
            "char_features_activas",
            "features_activas_total"
        ]
    ]
)

CONTRAFACTUAL — DATASCIENCE_005 ES


,variante,categoria_predicha,segunda_categoria,estado,margen_decision,word_features_activas,char_features_activas,features_activas_total
0,original,backend,cloud,revision,0.343937,5,258,263
1,mas_tecnico,datascience,cloud,aceptada,0.663672,3,211,214
2,terminologia_ml,datascience,cloud,aceptada,0.662851,2,221,223
3,ingles_tecnico,datascience,cloud,aceptada,0.851804,4,215,219


In [60]:
# ============================================================
# CELDA 45 — RESUMEN FINAL DEL BENCHMARK
# ============================================================

total_documentos = len(df_resultados)
total_correctas = int(df_resultados["correcta"].sum())
total_errores = total_documentos - total_correctas

accuracy_global = (
    total_correctas /
    total_documentos
)

total_casos = (
    df_resultados["case_id"]
    .nunique()
)

casos_consistentes = int(
    consistencia["consistente"].sum()
)

tasa_consistencia = (
    casos_consistentes /
    total_casos
)

errores_capturados = int(
    df_resultados[
        ~df_resultados["correcta"]
    ]["estado"]
    .isin([
        "revision",
        "rechazada"
    ])
    .sum()
)

tasa_captura_errores = (
    errores_capturados /
    total_errores
    if total_errores > 0
    else 1.0
)

aceptadas = (
    df_resultados[
        df_resultados["estado"]
        == "aceptada"
    ]
)

accuracy_aceptadas = (
    aceptadas["correcta"].mean()
    if len(aceptadas) > 0
    else None
)

print("=" * 70)
print("RESUMEN FINAL — BENCHMARK MULTILINGÜE")
print("=" * 70)

print(f"\nDocumentos:               {total_documentos}")
print(f"Casos semánticos:          {total_casos}")
print(f"Correctas:                 {total_correctas}")
print(f"Errores:                   {total_errores}")
print(f"Accuracy global:           {accuracy_global:.2%}")

print(
    f"Consistencia cross-lang:   "
    f"{tasa_consistencia:.2%}"
)

print(
    f"Captura de errores:        "
    f"{tasa_captura_errores:.2%}"
)

if accuracy_aceptadas is not None:
    print(
        f"Accuracy aceptadas:        "
        f"{accuracy_aceptadas:.2%}"
    )

RESUMEN FINAL — BENCHMARK MULTILINGÜE

Documentos:               80
Casos semánticos:          20
Correctas:                 69
Errores:                   11
Accuracy global:           86.25%
Consistencia cross-lang:   65.00%
Captura de errores:        100.00%
Accuracy aceptadas:        100.00%


In [61]:
# ============================================================
# CELDA 46 — EXPORTACIÓN DE RESULTADOS
# ============================================================

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Predicciones completas
# ------------------------------------------------------------

df_resultados.to_csv(
    REPORT_DIR
    / "multilingual_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Métricas por idioma
# ------------------------------------------------------------

df_metricas_idioma.to_csv(
    REPORT_DIR
    / "metrics_by_language.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Idioma × categoría
# ------------------------------------------------------------

tabla_idioma_categoria.to_csv(
    REPORT_DIR
    / "metrics_by_language_category.csv",
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Consistencia
# ------------------------------------------------------------

consistencia.to_csv(
    REPORT_DIR
    / "cross_language_consistency.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Taxonomía de errores
# ------------------------------------------------------------

df_errores_taxonomia.to_csv(
    REPORT_DIR
    / "error_taxonomy.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Features diferenciales
# ------------------------------------------------------------

df_differential.to_csv(
    REPORT_DIR
    / "differential_features.csv",
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("EXPORTACIÓN COMPLETADA")
print("=" * 70)

for archivo in sorted(
    REPORT_DIR.glob("*.csv")
):
    print("✅", archivo.name)

EXPORTACIÓN COMPLETADA
✅ cross_language_consistency.csv
✅ differential_features.csv
✅ error_taxonomy.csv
✅ metrics_by_language.csv
✅ metrics_by_language_category.csv
✅ multilingual_predictions.csv


In [62]:
# ============================================================
# CELDA 47 — MULTILINGUAL EVALUATION REPORT
# ============================================================

resumen_taxonomia = (
    df_errores_taxonomia[
        "failure_type"
    ]
    .value_counts()
    .to_dict()
)

reporte_multilingue = {

    "experiment": {
        "name":
            "TechMind Multilingual Pilot Benchmark",

        "model_version":
            "1.1.0",

        "api_interface":
            "1.0.0",

        "benchmark_type":
            "synthetic_controlled",

        "documents":
            int(total_documentos),

        "semantic_cases":
            int(total_casos),

        "languages": [
            "es",
            "en",
            "ru",
            "es_en"
        ],

        "categories": [
            "backend",
            "cloud",
            "datascience",
            "frontend"
        ]
    },

    "global_metrics": {
        "accuracy":
            float(accuracy_global),

        "correct_predictions":
            int(total_correctas),

        "errors":
            int(total_errores),

        "cross_language_consistency":
            float(tasa_consistencia),

        "error_capture_rate":
            float(tasa_captura_errores),

        "accepted_accuracy":
            (
                float(accuracy_aceptadas)
                if accuracy_aceptadas
                is not None
                else None
            )
    },

    "metrics_by_language":
        df_metricas_idioma
        .round(6)
        .to_dict(
            orient="records"
        ),

    "failure_taxonomy":
        {
            str(k): int(v)
            for k, v
            in resumen_taxonomia.items()
        },

    "findings": [
        (
            "English and Spanish-English mixed "
            "content achieved 95% accuracy."
        ),
        (
            "Russian achieved 80% accuracy but "
            "no predictions were operationally accepted."
        ),
        (
            "All benchmark errors were routed to "
            "review or rejection."
        ),
        (
            "Cross-language prediction consistency "
            "was 65%."
        ),
        (
            "Russian datascience content showed a "
            "systematic datascience-to-cloud confusion."
        ),
        (
            "Counterfactual tests confirmed lexical "
            "sensitivity in selected Spanish examples."
        )
    ],

    "limitations": [
        (
            "Pilot benchmark contains only 20 "
            "semantic cases."
        ),
        (
            "Examples are synthetic and controlled."
        ),
        (
            "Results do not establish official "
            "multilingual support."
        ),
        (
            "Counterfactual examples are diagnostic "
            "and are not included in benchmark metrics."
        )
    ]
}

PATH_JSON = (
    REPORT_DIR
    / "multilingual_evaluation_report.json"
)

with open(
    PATH_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        reporte_multilingue,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "✅ Reporte JSON:",
    PATH_JSON
)

✅ Reporte JSON: C:\Users\MAMÁ\Downloads\techmind-v2\reports\multilingual\multilingual_evaluation_report.json


In [63]:
# ============================================================
# CELDA 48 — EXPORTAR EXPERIMENTOS CONTRAFACTUALES
# ============================================================

df_cf_cloud.to_csv(
    REPORT_DIR
    / "counterfactual_cloud_005.csv",
    index=False,
    encoding="utf-8-sig"
)

df_cf_ds.to_csv(
    REPORT_DIR
    / "counterfactual_datascience_005_es.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "✅ Contrafactuales exportados "
    "por separado."
)

✅ Contrafactuales exportados por separado.
